In [1]:
# Upgrade pip first
!pip install --upgrade pip

# Core ML libraries
!pip install torch==2.1.0+cu121 torchvision==0.16.1+cu121 torchaudio==2.1.1 --extra-index-url https://download.pytorch.org/whl/cu121

# Hugging Face and tokenizer tools
!pip install transformers==4.34.0 sentencepiece tokenizers safetensors

# Efficient fine-tuning
!pip install peft bitsandbytes accelerate einops

# Dataset and evaluation
!pip install datasets evaluate

# Experiment tracking (optional but recommended)
!pip install wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 20.9 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu121
ERROR: Could not find a version that satisfies the requirement torch==2.1.0+cu121 (from versions: 2.2.0, 2.2.0+cu121, 2.2.1, 2.2.1+cu121, 2.2.2, 2.2.2+cu121, 2.3.0, 2.3.0+cu121, 2.3.1, 2.3.1+cu121, 2.4.0, 2.4.0+cu121, 2.4.1, 2.4.1+cu121, 2.5.0, 2.5.0+cu121, 2.5.1, 2.5.1+cu121, 2.6.0, 2.7.0, 2.7.1, 2.8.0)
ERROR: No matching distribution found for torch==2.1.0+cu121
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 102.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 109.6 MB/s  0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.34.4
    Uninstalling huggingface-hub-0.34.4:
      Successfully uninstalled huggingface

In [2]:
import pandas as pd
from datasets import Dataset

# Load CSV with pandas
df = pd.read_csv("/content/finetune_dataset.csv")
print(df.head())

# Converting it to Hugging Face Dataset
dataset = Dataset.from_pandas(df)
print(dataset)


                                          input_text  \
0  Context: Since our original focus on PC graphi...   
1  Context: Some of the most recent applications ...   
2  Context: Our invention of the GPU in 1999 defi...   
3  Context: NVIDIA has a platform strategy, bring...   
4  Context: With our introduction of the CUDA pro...   

                                               label  
0           NVIDIA initially focused on PC graphics.  
1  Recent applications of GPU-powered deep learni...  
2                   NVIDIA invented the GPU in 1999.  
3  NVIDIA's platform strategy brings together har...  
4  NVIDIA's CUDA programming model opened the par...  
Dataset({
    features: ['input_text', 'label'],
    num_rows: 7000
})


In [3]:

from datasets import load_dataset

# Load CSV
dataset = load_dataset("csv", data_files="/content/finetune_dataset.csv", split="train")

# drop rows with missing/empty fields coerce to str
def _valid(row):
    it = row.get("input_text")
    lb = row.get("label")
    return (it is not None) and (lb is not None) and (str(it).strip() != "") and (str(lb).strip() != "")

dataset = dataset.filter(_valid)
dataset = dataset.map(lambda r: {"input_text": str(r["input_text"]), "label": str(r["label"])})

len(dataset), dataset[0]

Generating train split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/7000 [00:00<?, ? examples/s]

Map:   0%|          | 0/6998 [00:00<?, ? examples/s]

(6998,
 {'input_text': 'Context: Since our original focus on PC graphics, we have expanded to several other large and important computationally intensive fields. Question: What area did NVIDIA initially focus on before expanding to other computationally intensive fields?',
  'label': 'NVIDIA initially focused on PC graphics.'})

In [4]:
split_ds = dataset.train_test_split(test_size=0.1, seed=42)
train_ds = split_ds["train"]
val_ds = split_ds["test"]

print("Train samples:", len(train_ds))
print("Validation samples:", len(val_ds))
print("Columns:", train_ds.column_names)
print("Sample:", train_ds[0])

Train samples: 6298
Validation samples: 700
Columns: ['input_text', 'label']
Sample: {'input_text': 'Context: •Level 3 – inputs are generally unobservable and typically reflect management’s estimates of assumptions that market participants would use in pricing the asset or liability. The fair values are therefore determined using model-based techniques, including option pricing models and discounted cash flow models. Question: What type of inputs are typically reflected in Level 3 assets and liabilities valuations?', 'label': "Level 3 asset and liability valuations typically reflect management's estimates of assumptions that market participants would use in pricing the asset or liability."}


In [7]:
from transformers import AutoTokenizer, default_data_collator

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
MAX_LEN = 1024

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

def preprocess(example):
    user_text = (example["input_text"] or "").strip()
    answer_text = (example["label"] or "").strip()

    messages_full = [
        {"role": "user", "content": user_text},
        {"role": "assistant", "content": answer_text},
    ]
    messages_prompt_only = [{"role": "user", "content": user_text}]

    prompt_text = tokenizer.apply_chat_template(
        messages_prompt_only, tokenize=False, add_generation_prompt=True
    )
    full_text = tokenizer.apply_chat_template(
        messages_full, tokenize=False, add_generation_prompt=False
    )

    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    full_ids   = tokenizer(full_text,  add_special_tokens=False)["input_ids"]

    # Guard rails
    if not full_ids or len(full_ids) < len(prompt_ids):
        return None  # drop

    if full_ids[:len(prompt_ids)] != prompt_ids:
        found = -1
        for i in range(0, max(0, len(full_ids) - len(prompt_ids)) + 1):
            if full_ids[i:i+len(prompt_ids)] == prompt_ids:
                found = i
                break
        if found == -1:
            return None  # drop
        split_at = found + len(prompt_ids)
    else:
        split_at = len(prompt_ids)

    labels = [-100] * split_at + full_ids[split_at:]

    # Truncate
    input_ids = full_ids[:MAX_LEN]
    labels    = labels[:MAX_LEN]
    attention_mask = [1] * len(input_ids)

    # Drop if answer fully truncated
    if not input_ids or all(l == -100 for l in labels):
        return None

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

cols_to_remove = ["input_text", "label"]

train_ds_tok = train_ds.map(preprocess, remove_columns=cols_to_remove, desc="tokenize/train")
val_ds_tok = val_ds.map(preprocess, remove_columns=cols_to_remove, desc="tokenize/val")

# Extra safety: remove any empties
train_ds_tok = train_ds_tok.filter(lambda ex: len(ex["input_ids"]) > 0 and any(l != -100 for l in ex["labels"]))
val_ds_tok   = val_ds_tok.filter(lambda ex: len(ex["input_ids"]) > 0 and any(l != -100 for l in ex["labels"]))

print("train/val sizes:", len(train_ds_tok), len(val_ds_tok))
print("sample lens:", len(train_ds_tok[0]["input_ids"]), len(train_ds_tok[0]["labels"]))


tokenize/train:   0%|          | 0/6298 [00:00<?, ? examples/s]

tokenize/val:   0%|          | 0/700 [00:00<?, ? examples/s]

Filter:   0%|          | 0/6297 [00:00<?, ? examples/s]

Filter:   0%|          | 0/700 [00:00<?, ? examples/s]

train/val sizes: 6297 700
sample lens: 109 109


In [8]:

## QLoRA finetune with mistral model loading

import os, math, torch
from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, get_peft_model

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
OUTPUT_DIR = "mistral7b_v03_qlora_finance"

def has_flash_attn():
    try:
        import flash_attn  # noqa
        return True
    except Exception:
        return False

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,           # 8-bit mode
    bnb_8bit_compute_dtype=torch.bfloat16
)

# Load the base model quantized with nf4 datatype
attn_impl = "flash_attention_2" if has_flash_attn() else "eager"
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float16,
    attn_implementation=attn_impl,
)

# Important for gradient checkpointing later
model.config.use_cache = False

# LoRA config — fairly beefy but still memory-friendly
lora_config = LoraConfig(
    r=128,                         # 16 performeed better in my previous experiment with LLaMA model
    lora_alpha=32,                # scaling
    lora_dropout=0.1,            # regularization
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[              # attention + MLP finetuning from my previous observation
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_rslora=True,
)

model = get_peft_model(model, lora_config)
print(model.print_trainable_parameters())

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.55G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

trainable params: 335,544,320 || all params: 7,583,567,872 || trainable%: 4.4246
None


In [9]:

import wandb
wandb.login()
os.environ["WANDB_PROJECT"] = "mistral-finance-sft"
RUN_NAME = "mistral7b_v03_qlora_teacherforcing_2"


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: abhi1199 (abhi1199-city-university-of-london) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [10]:

import os, math, torch, wandb
from transformers import TrainingArguments, Trainer, TrainerCallback

# W&B
wandb.login()
os.environ["WANDB_PROJECT"] = "mistral-finance-sft"
RUN_NAME = "mistral7b_v03_qlora_teacherforcing_5"



In [11]:

from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)



if hasattr(model, "_orig_mod"):
    # unwrap compiled model so Trainer accepts it for PEFT fine-tuning
    model = model._orig_mod


# Trainer
from transformers import TrainingArguments, Trainer
import math, wandb, torch

args = TrainingArguments(
    output_dir="mistral7b_v03_qlora_out",
    run_name="mistral7b_v03_qlora_teacherforcing",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    weight_decay=0.05,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    max_grad_norm=1.0,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=torch.cuda.is_available(),
    fp16=not torch.cuda.is_available(),
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to=["wandb"],
    remove_unused_columns=False,
)


class PerplexityPrinter(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs: return
        if "loss" in logs:
            loss = float(logs["loss"])
            ppl = math.exp(loss) if loss < 20 else float("inf")
            print(f"[train] step {int(state.global_step)} - loss {loss:.4f} - ppl {ppl:.2f}")
            wandb.log({"train/loss": loss, "train/perplexity": ppl, "step": int(state.global_step)})

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if not metrics or "eval_loss" not in metrics: return
        eval_loss = float(metrics["eval_loss"])
        eval_ppl = math.exp(eval_loss) if eval_loss < 20 else float("inf")
        print(f"[eval]  step {int(state.global_step)} - eval_loss {eval_loss:.4f} - eval_ppl {eval_ppl:.2f}")
        wandb.log({"eval/loss": eval_loss, "eval/perplexity": eval_ppl, "step": int(state.global_step)})

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds_tok,
    eval_dataset=val_ds_tok,
    processing_class=tokenizer,
    data_collator=data_collator,
    callbacks=[PerplexityPrinter()],
)

train_result = trainer.train()

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:186: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss,Validation Loss
200,0.853700,0.741489
400,0.756900,0.807190
600,0.509600,0.753004
800,0.253700,0.719463
1000,0.132500,0.695950


[train] step 20 - loss 0.5855 - ppl 1.80
[train] step 40 - loss 0.4508 - ppl 1.57
[train] step 60 - loss 0.5147 - ppl 1.67
[train] step 80 - loss 0.9111 - ppl 2.49
[train] step 100 - loss 0.6638 - ppl 1.94
[train] step 120 - loss 0.6952 - ppl 2.00
[train] step 140 - loss 0.7394 - ppl 2.09
[train] step 160 - loss 0.7643 - ppl 2.15
[train] step 180 - loss 0.7242 - ppl 2.06
[train] step 200 - loss 0.8537 - ppl 2.35
[eval]  step 200 - eval_loss 0.7415 - eval_ppl 2.10


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:186: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


[train] step 220 - loss 0.7870 - ppl 2.20
[train] step 240 - loss 0.8249 - ppl 2.28
[train] step 260 - loss 0.9096 - ppl 2.48
[train] step 280 - loss 0.8839 - ppl 2.42
[train] step 300 - loss 1.0097 - ppl 2.74
[train] step 320 - loss 0.8947 - ppl 2.45
[train] step 340 - loss 0.8512 - ppl 2.34
[train] step 360 - loss 0.8927 - ppl 2.44
[train] step 380 - loss 0.8558 - ppl 2.35
[train] step 400 - loss 0.7569 - ppl 2.13
[eval]  step 400 - eval_loss 0.8072 - eval_ppl 2.24


...

[train] step 620 - loss 0.5337 - ppl 1.71
[train] step 640 - loss 0.4842 - ppl 1.62
[train] step 660 - loss 0.4671 - ppl 1.60
[train] step 680 - loss 0.5122 - ppl 1.67
[train] step 700 - loss 0.5035 - ppl 1.65
[train] step 720 - loss 0.4275 - ppl 1.53
[train] step 740 - loss 0.4533 - ppl 1.57
[train] step 760 - loss 0.4612 - ppl 1.59
[train] step 780 - loss 0.4580 - ppl 1.58
[train] step 800 - loss 0.2537 - ppl 1.29
[eval]  step 800 - eval_loss 0.7195 - eval_ppl 2.05


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:186: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


[train] step 820 - loss 0.1568 - ppl 1.17
[train] step 840 - loss 0.1471 - ppl 1.16
[train] step 860 - loss 0.1556 - ppl 1.17
[train] step 880 - loss 0.1385 - ppl 1.15
[train] step 900 - loss 0.1803 - ppl 1.20
[train] step 920 - loss 0.1306 - ppl 1.14
[train] step 940 - loss 0.1518 - ppl 1.16
[train] step 960 - loss 0.1415 - ppl 1.15
[train] step 980 - loss 0.1393 - ppl 1.15
[train] step 1000 - loss 0.1325 - ppl 1.14
[eval]  step 1000 - eval_loss 0.6960 - eval_ppl 2.01


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:186: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


[train] step 1020 - loss 0.1344 - ppl 1.14
[train] step 1040 - loss 0.1142 - ppl 1.12
[train] step 1060 - loss 0.1113 - ppl 1.12
[train] step 1080 - loss 0.1268 - ppl 1.14
[train] step 1100 - loss 0.1011 - ppl 1.11
[train] step 1120 - loss 0.1069 - ppl 1.11
[train] step 1140 - loss 0.1226 - ppl 1.13
[train] step 1160 - loss 0.1339 - ppl 1.14
[train] step 1180 - loss 0.1361 - ppl 1.15


In [12]:

trainer.save_model()
tokenizer.save_pretrained(args.output_dir)


('mistral7b_v03_qlora_out/tokenizer_config.json',
 'mistral7b_v03_qlora_out/special_tokens_map.json',
 'mistral7b_v03_qlora_out/chat_template.jinja',
 'mistral7b_v03_qlora_out/tokenizer.model',
 'mistral7b_v03_qlora_out/added_tokens.json',
 'mistral7b_v03_qlora_out/tokenizer.json')

In [13]:

# --- Final eval + print + log ---
metrics = trainer.evaluate()
if "eval_loss" in metrics:
    try:
        eval_ppl = math.exp(metrics["eval_loss"])
    except OverflowError:
        eval_ppl = float("inf")
    print(f"[final] eval_loss {metrics['eval_loss']:.4f} - eval_ppl {eval_ppl:.2f}")
    wandb.log({"final/eval_loss": metrics["eval_loss"], "final/eval_perplexity": eval_ppl})
print(metrics)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:186: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


[eval]  step 1182 - eval_loss 0.6960 - eval_ppl 2.01
[final] eval_loss 0.6960 - eval_ppl 2.01
{'eval_loss': 0.6959500908851624, 'eval_runtime': 114.2814, 'eval_samples_per_second': 6.125, 'eval_steps_per_second': 3.063, 'epoch': 3.0}


In [16]:
from peft import PeftModel

BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"
ADAPTER_PATH = "/content/mistral7b_v03_qlora_out"

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# Base model in 8-bit with same config
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float16
)

# Load LoRA adapter
model = PeftModel.from_pretrained(model, ADAPTER_PATH)
model.eval()

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32768, 4096)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): lora.Linear8bitLt(
                (base_layer): Linear8bitLt(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=128, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=128, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (

In [17]:
import torch

def ask_question(model, tokenizer, question, context=None, max_new_tokens=200, temperature=0.7):
    """
    Ask a question to a model (base + LoRA).

    Args:
        model: Loaded model (PeftModel)
        tokenizer: Corresponding tokenizer
        question: The question string
        context: Optional context string
        max_new_tokens: Max tokens to generate
        temperature: Sampling temperature
    Returns:
        Generated answer as a string
    """
    if context:
        prompt = f"Context:\n{context}\n\nQuestion:\n{question}\nAnswer:"
    else:
        prompt = f"Question:\n{question}\nAnswer:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    answer = tokenizer.decode(output[0], skip_special_tokens=True)
    # Remove prompt from output
    answer = answer.replace(prompt, "").strip()
    return answer


In [18]:
question = "The Company allocates the transaction price to each performance obligation on a relative SSP basis. Judgment is required to determine the SSP for each distinct performance obligation. The Company determines SSP by considering its overall pricing objectives and market conditions. Significant pricing practices taken into consideration include the Company’s discounting practices, the size and volume of the Company’s transactions, the customer demographic, the geographic area where services are sold, price lists, the Company's go-to-market strategy, historical and current sales and contract prices."
context = "What is the basis for the Company to determine the Standalone Selling Price (SSP) for each distinct performance obligation in contracts with multiple performance obligations?"

answer = ask_question(model, tokenizer, question, context)
print("Answer:", answer)

Answer: The Company determines the Standalone Selling Price (SSP) by considering its overall pricing objectives and market conditions.


In [19]:

context = "As the rate implicit in the lease is rarely readily determinable, Delta Air Lines uses their incremental borrowing rate, which is based on the estimated interest rate for collateralized borrowing over a similar term of the lease at commencement date."
question = "What discount rate does Delta Air Lines use for lease payments when the rate implicit in the lease is not readily determinable?"
answer = ask_question(model, tokenizer, question, context)
print("Answer:", answer)

Answer: Delta Air Lines uses their incremental borrowing rate, which is based on the estimated interest rate for collateralized borrowing over a similar term of the lease at commencement date.


In [20]:
context = ""
question = "What business segment of AT&T focuses on delivering nationwide wireless service and equipment?"
answer = ask_question(model, tokenizer, question, context)
print("Answer:", answer)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:186: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Answer: The nationwide wireless service and equipment segment of AT&T focuses on providing nationwide wireless service and equipment.


In [21]:
context = "Mobility, one of the business units within the Communications segment, provides nationwide wireless service and equipment"
question = "What business segment of AT&T focuses on delivering nationwide wireless service and equipment?"
answer = ask_question(model, tokenizer, question, context)
print("Answer:", answer)

Answer: The Mobility business segment of AT&T focuses on providing nationwide wireless service and equipment.


In [22]:
context = ""
question = "What business segment of AT&T focuses on delivering nationwide wireless service and equipment?"
answer = ask_question(model, tokenizer, question, context)
print("Answer:", answer)

Answer: AT&T focuses on delivering nationwide wireless service and equipment.


In [23]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, PeftConfig

BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"
ADAPTER_PATH = "/content/mistral7b_v03_qlora_out"
MERGED_MODEL_DIR = "./mistral7b_merged"

# Load base model
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float16
)

# Load LoRA adapter
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

# Merge LoRA weights into base model
merged_model = model.merge_and_unload()


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:93: UserWarning: Merge lora module to 8-bit linear may get different generations due to rounding errors.
  warnings.warn(


In [24]:
# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.save_pretrained(MERGED_MODEL_DIR)

# Merged model
merged_model.save_pretrained(MERGED_MODEL_DIR)


In [38]:
from huggingface_hub import login

# Paste your newly created write token here
login(token="")


{'type': 'user', 'id': '6735f3017e5caf2c6177235e', 'name': 'iamAbhishek01', 'fullname': 'Abhishek Das', 'email': 'dummy1cx@gmail.com', 'emailVerified': True, 'canPay': False, 'periodEnd': None, 'isPro': False, 'avatarUrl': '/avatars/687f3443c12597972b1076e6a0c33054.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'mistral', 'role': 'write', 'createdAt': '2025-08-29T03:51:17.422Z'}}}


In [28]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained(MERGED_DIR)
tok   = AutoTokenizer.from_pretrained(MERGED_DIR)

model.push_to_hub(REPO_ID, token=WRITE_TOKEN)
tok.push_to_hub(REPO_ID,   token=WRITE_TOKEN)
print("Pushed:", REPO_ID)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  ...yy/model-00002-of-00002.safetensors:   2%|1         | 41.9MB / 2.57GB            

  ...yy/model-00001-of-00002.safetensors:   1%|          | 41.9MB / 4.95GB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  /tmp/tmpr0tw6vl9/tokenizer.model      : 100%|##########|  587kB /  587kB            

Pushed: iamAbhishek01/mistral7b-finance-merged


In [29]:
from transformers import AutoModelForCausalLM, AutoTokenizer
tok2 = AutoTokenizer.from_pretrained(REPO_ID)
mdl2 = AutoModelForCausalLM.from_pretrained(REPO_ID, device_map="auto", torch_dtype="auto")
print("Loaded OK")


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.57G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Loaded OK


In [30]:
import torch

def ask_question(model, tokenizer, question, context=None, max_new_tokens=200, temperature=0.7):
    """
    Ask a question to your fine-tuned model with optional context.
    """
    if context:
        prompt = f"Context:\n{context}\n\nQuestion:\n{question}\nAnswer:"
    else:
        prompt = f"Question:\n{question}\nAnswer:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Strip the prompt to leave only the model’s answer
    answer = text.replace(prompt, "").strip()
    return answer
